# Agentic AI Performance Classification
### K-Means Clustering + Logistic Regression
**Student:** Richard Clay | A23CS0342
**Supervisor:** Dr. Aryati Binti Bakri | SECP3843-01

---
## 0. Setup & Load Gold Data

In [1]:
# ── Setup: pandas + scikit-learn ──
import os, warnings, json
warnings.filterwarnings('ignore')
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV

import pyarrow.parquet as pq

BASE_DIR = Path(r'C:\Users\richa\individual-project')
GOLD_DIR = BASE_DIR / '05_Output_Gold'

print('Libraries loaded. Ready.')

Libraries loaded. Ready.


---
## 1. K-Means Clustering (Unsupervised)
3 clusters based on: Task_Success_Rate, Productivity_Improvement_Percent, Leadership_Trust_Score

In [2]:
# Load Gold parquet → pandas DataFrame
df = pq.read_table(str(GOLD_DIR / 'ai_enriched_agentic_leadership')).to_pandas()
print(f'Loaded {len(df):,} records from Gold layer')
print(f'Columns: {list(df.columns)}')

Loaded 5,500 records from Gold layer
Columns: ['Record_ID', 'Industry', 'Organization_Size', 'Leadership_Function', 'AI_Maturity_Level', 'Agent_Type', 'Use_Case_Area', 'Agent_Autonomy_Level', 'Decision_Making_Type', 'Context_Awareness_Score', 'Task_Complexity_Level', 'Human_Oversight_Level', 'Explainability_Level', 'Data_Privacy_Compliance', 'Integration_Level', 'Task_Success_Rate', 'Response_Time_Seconds', 'Productivity_Improvement_Percent', 'Leadership_Trust_Score', 'Adoption_Success_Level', 'ingestion_date', 'data_source', 'Benchmark_Task_Success_Rate', 'Benchmark_Productivity_Improvement', 'Benchmark_Trust_Score', 'Execution_Time_Minutes', 'Error_Count', 'Mapped_Sector', 'Sector_Avg_Citation', 'Productivity_Category', 'Trust_vs_Benchmark_Gap']


---
## 2. Logistic Regression (Supervised)
Predict `Productivity_Category` (low/medium/high) using **system features only** — no clustering leakage.

In [3]:
# ── 1. K-Means Clustering (sklearn) ──
km_features = ['Task_Success_Rate', 'Productivity_Improvement_Percent', 'Leadership_Trust_Score']

# Drop rows with nulls in these features
df_km = df[km_features].dropna().copy()

scaler = StandardScaler()
X_km = scaler.fit_transform(df_km)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_km)
sil_score = silhouette_score(X_km, clusters)
print(f'Silhouette Score: {sil_score:.4f}')

# Label clusters by productivity centroid
centroids_scaled = kmeans.cluster_centers_
prod_idx = km_features.index('Productivity_Improvement_Percent')
order = sorted(range(3), key=lambda i: centroids_scaled[i][prod_idx])
label_map = {order[0]: 'Low Performer', order[1]: 'Average Performer', order[2]: 'High Performer'}
df_km['Cluster_Label'] = [label_map[c] for c in clusters]

print('Cluster mapping:', label_map)
print('\nCluster distribution:')
print(df_km['Cluster_Label'].value_counts())

# Centroid summary in original scale
centroid_orig = scaler.inverse_transform(kmeans.cluster_centers_)
print('\nCentroids (original scale):')
for i, label in label_map.items():
    print(f'  {label}: Success={centroid_orig[i][0]:.2f}, Productivity={centroid_orig[i][1]:.2f}, Trust={centroid_orig[i][2]:.2f}')

Silhouette Score: 0.3489
Cluster mapping: {2: 'Low Performer', 0: 'Average Performer', 1: 'High Performer'}

Cluster distribution:
Cluster_Label
High Performer       2362
Low Performer        1824
Average Performer    1314
Name: count, dtype: int64

Centroids (original scale):
  Low Performer: Success=68.43, Productivity=8.21, Trust=70.09
  Average Performer: Success=64.84, Productivity=9.86, Trust=44.35
  High Performer: Success=82.77, Productivity=17.85, Trust=66.17


---
## 3. Visual Analytics (Matplotlib + Seaborn)
Export charts untuk presentasi — tidak perlu plot di notebook, tinggal simpan ke `docs/report_images/`.

In [4]:
# ── 2. Logistic Regression (sklearn) ──
lr_features = ['Context_Awareness_Score', 'Response_Time_Seconds', 'Task_Complexity_Level', 'Adoption_Success_Level', 'Error_Count']

df_lr = df.dropna(subset=lr_features + ['Productivity_Category']).copy()
df_lr['label'] = LabelEncoder().fit_transform(df_lr['Productivity_Category'])  # low=0, medium=1, high=2

# Encode categorical features
df_lr['Task_Complexity_Level'] = LabelEncoder().fit_transform(df_lr['Task_Complexity_Level'].astype(str))
df_lr['Adoption_Success_Level'] = LabelEncoder().fit_transform(df_lr['Adoption_Success_Level'].astype(str))

# Fill remaining nulls in Error_Count
df_lr['Error_Count'] = df_lr['Error_Count'].fillna(0.0)

X = df_lr[lr_features].values.astype(np.float64)
y = df_lr['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

# GridSearch
lr = LogisticRegression(max_iter=200, random_state=42)
param_grid = {'C': [10, 20]}  # C = 1/regParam
grid = GridSearchCV(lr, param_grid, cv=3, scoring='accuracy')
grid.fit(X_train, y_train)

best_lr = grid.best_estimator_
print(f'Best C: {grid.best_params_["C"]}')

y_pred = best_lr.predict(X_test)
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 (macro): {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred, average="weighted"):.4f}')
print(f'\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Train: 4,400 | Test: 1,100


Best C: 20
Accuracy : 0.6091
F1 (macro): 0.5939
Precision: 0.6150
Recall   : 0.6091

Confusion Matrix:
[[304  24 117]
 [ 30 117 107]
 [ 91  61 249]]


In [5]:
# ── 3. Visual Analytics ──
os.makedirs('docs/report_images', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Chart 1: Cluster Pie + Centroids
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
cluster_counts = df_km['Cluster_Label'].value_counts().reindex(['High Performer', 'Average Performer', 'Low Performer'])
colors = ['#2ca02c', '#1f77b4', '#d62728']
ax[0].pie(cluster_counts, labels=cluster_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax[0].set_title('Cluster Distribution')
x = np.arange(3)
centroid_df = df_km.groupby('Cluster_Label')[km_features].mean().reindex(['Low Performer', 'Average Performer', 'High Performer'])
width = 0.25
ax[1].bar(x - width, centroid_df['Task_Success_Rate'], width, label='Success Rate', color='#2ca02c')
ax[1].bar(x, centroid_df['Productivity_Improvement_Percent'], width, label='Productivity %', color='#1f77b4')
ax[1].bar(x + width, centroid_df['Leadership_Trust_Score'], width, label='Trust Score', color='#d62728')
ax[1].set_xticks(x)
ax[1].set_xticklabels(centroid_df.index)
ax[1].set_ylabel('Score')
ax[1].set_title('Cluster Centroids')
ax[1].legend()
plt.tight_layout()
plt.savefig('docs/report_images/cluster_analysis.png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved: cluster_analysis.png')

# Chart 2: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Logistic Regression Confusion Matrix')
plt.tight_layout()
plt.savefig('docs/report_images/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved: confusion_matrix.png')

# Chart 3: Trust Gap by Industry
df_full = df.copy()
industry_gap = df_full.groupby('Industry')['Trust_vs_Benchmark_Gap'].mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#d62728' if x < 0 else '#2ca02c' for x in industry_gap]
industry_gap.plot(kind='barh', color=colors_bar, ax=ax)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Trust Gap (Internal - Benchmark)')
ax.set_title('Trust Gap by Industry (Positive = Trust > Benchmark)')
plt.tight_layout()
plt.savefig('docs/report_images/trust_gap_by_industry.png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved: trust_gap_by_industry.png')

# Chart 4: Feature Importance (LR Coefficients)
feat_names = ['Context_Awareness', 'Response_Time', 'Task_Level', 'Adoption_Success', 'Error_Count']
coeff = best_lr.coef_
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(feat_names))
width = 0.25
for i, cls in enumerate(['Low', 'Medium', 'High']):
    ax.bar(x + (i-1)*width, coeff[i], width, label=cls)
ax.set_xticks(x)
ax.set_xticklabels(feat_names, rotation=15)
ax.set_ylabel('Coefficient')
ax.set_title('Logistic Regression Coefficients per Class')
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('docs/report_images/lr_feature_importance.png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved: lr_feature_importance.png')
print('\n✅ Classification complete — all charts saved to docs/report_images/')

Saved: cluster_analysis.png


Saved: confusion_matrix.png


Saved: trust_gap_by_industry.png


Saved: lr_feature_importance.png

✅ Classification complete — all charts saved to docs/report_images/
